# Step 7-3b: 감성 분류 (KcELECTRA, 두 트랙)

7-3a에서 선정된 **KcELECTRA-multi** 모델로 두 트랙 동시 실행.

## 트랙 구성

| 트랙 | 입력 | 단위 | 용도 |
|---|---|---|---|
| **A. ABSA 메인** | `step7_2_aspect_pairs.parquet` | 문장 × 측면 | 측면별 긍/중/부 |
| **B. 리뷰 단위 보조** | `embedding_input.parquet` | 리뷰 전체 | 매칭 실패 보완 + 별점 교차검증 |

## 신뢰도 기반 중립 (옵션 C)

```
모델 confidence (긍/부 매핑 후 합산 확률)
  > THRESHOLD_HIGH  → 모델 예측 그대로
  < THRESHOLD_LOW   → neutral (모델도 확신 못함)
  그 외             → 모델 예측 그대로
```

Threshold는 분할 문장 1000건 샘플의 confidence 분포를 보고 자동 결정 (분위수 기반).

## 입력 / 출력
| 파일 | 내용 |
|---|---|
| `step7_2_aspect_pairs.parquet` | (문장, 측면) 156만 페어 |
| `embedding_input.parquet` | 정제된 리뷰 62만 건 |
| → `step7_3b_thresholds.json` | 결정된 threshold 저장 |
| → `step7_3b_aspect_sentiment.parquet` | ABSA 결과 (메인) |
| → `step7_3b_review_sentiment.parquet` | 리뷰 단위 감성 (보조) |

## 예상 실행 시간 (T4/L4 GPU 기준)
- Threshold 결정 (1000건): ~2분
- ABSA 메인 (156만): ~3-5시간
- 리뷰 단위 (62만): ~1-2시간
- **합계: 4-7시간**


## 0. 환경 셋업

In [20]:
!pip install -q transformers accelerate

In [21]:
from google.colab import drive

import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

In [22]:
drive.mount('/content/drive')

OUTPUT_DIR  = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
PAIRS_PATH  = OUTPUT_DIR + 'step7_2_aspect_pairs.parquet'
REVIEW_PATH = OUTPUT_DIR + 'embedding_input.parquet'

print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'PAIRS_PATH : {PAIRS_PATH}')
print(f'REVIEW_PATH: {REVIEW_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUTPUT_DIR : /content/drive/MyDrive/sparta/tp4/review_analysis/data/
PAIRS_PATH : /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_2_aspect_pairs.parquet
REVIEW_PATH: /content/drive/MyDrive/sparta/tp4/review_analysis/data/embedding_input.parquet


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


## 7-3b-1. 모델 로드 + 라벨 매핑

7-3a에서 검증된 매핑 그대로 사용.

In [24]:
# KcELECTRA 11 라벨 → 3-class 매핑 (7-3a에서 검증된 설정)
KCELECTRA_POSITIVE = {
    '기쁨(행복한)', '고마운', '설레는(기대하는)',
    '사랑하는', '즐거운(신나는)',
}
KCELECTRA_NEGATIVE = {
    '슬픔(우울한)', '힘듦(지침)', '짜증남', '걱정스러운(불안한)',
}
KCELECTRA_NEUTRAL = {
    '일상적인', '생각이 많은',
}

# 모델 로드
MODEL_ID = 'nlp04/korean_sentiment_analysis_kcelectra'
print(f'로드: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

# 라벨 → 카테고리 인덱스 매핑 (벡터화된 추론용)
id2label = model.config.id2label
POS_IDX = [i for i, lab in id2label.items() if lab in KCELECTRA_POSITIVE]
NEG_IDX = [i for i, lab in id2label.items() if lab in KCELECTRA_NEGATIVE]
NEU_IDX = [i for i, lab in id2label.items() if lab in KCELECTRA_NEUTRAL]

print(f'\n라벨 인덱스 매핑:')
print(f'  Positive ({len(POS_IDX)}): {[id2label[i] for i in POS_IDX]}')
print(f'  Negative ({len(NEG_IDX)}): {[id2label[i] for i in NEG_IDX]}')
print(f'  Neutral  ({len(NEU_IDX)}): {[id2label[i] for i in NEU_IDX]}')

# 검증 — 11개 모두 매핑됐는지
all_idx = set(POS_IDX) | set(NEG_IDX) | set(NEU_IDX)
unmapped = set(id2label.keys()) - all_idx
assert len(unmapped) == 0, f'⚠️ 매핑 누락: {[id2label[i] for i in unmapped]}'
print(f'\n[11개 라벨 모두 매핑 완료]')

로드: nlp04/korean_sentiment_analysis_kcelectra


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: nlp04/korean_sentiment_analysis_kcelectra
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



라벨 인덱스 매핑:
  Positive (5): ['기쁨(행복한)', '고마운', '설레는(기대하는)', '사랑하는', '즐거운(신나는)']
  Negative (4): ['슬픔(우울한)', '힘듦(지침)', '짜증남', '걱정스러운(불안한)']
  Neutral  (2): ['일상적인', '생각이 많은']

[11개 라벨 모두 매핑 완료]


## 7-3b-2. 배치 추론 함수

### 출력 구조
한 텍스트 입력당:
- `p_pos`: 긍정 카테고리 5개 라벨 확률 합
- `p_neg`: 부정 카테고리 4개 라벨 확률 합
- `p_neu`: 중립 카테고리 2개 라벨 확률 합
- (이 셋의 합 = 1.0)

이후 신뢰도 기반 중립 규칙으로 최종 라벨 결정.

In [25]:
@torch.no_grad()
def predict_sentiment_probs(texts, batch_size=64, max_length=128):
    """감성 확률 (긍/부/중) 배치 추론.

    Returns:
        np.ndarray of shape (N, 3): [p_pos, p_neg, p_neu]
    """
    all_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc='추론'):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=max_length, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()  # (B, 11)

        # 카테고리별 확률 합산
        p_pos = probs[:, POS_IDX].sum(axis=1)
        p_neg = probs[:, NEG_IDX].sum(axis=1)
        p_neu = probs[:, NEU_IDX].sum(axis=1)

        all_probs.append(np.stack([p_pos, p_neg, p_neu], axis=1))

    return np.concatenate(all_probs, axis=0)


# ─────────────────────────────────────────────
# 함수 검증 — 10건 테스트
# ─────────────────────────────────────────────
test_texts = [
    '사이즈가 딱 맞아요',
    '너무 작아서 입을 수 없어요',
    '색감 예뻐요',
    '배송이 늦어서 짜증나요',
    '재질은 괜찮은 편이에요',
    '실밥이 풀려있고 봉제가 엉망이에요',
    '기모가 따뜻해요',
    '비싸고 별로예요',
    '잘 받았어요',
    '이쁘고 가성비도 좋아요',
]

probs = predict_sentiment_probs(test_texts, batch_size=10)
print(f'\n[샘플 추론]')
print(f'{"text":<40} {"p_pos":>6} {"p_neg":>6} {"p_neu":>6}')
print('-' * 65)
for t, p in zip(test_texts, probs):
    print(f'{t[:38]:<40} {p[0]:>6.3f} {p[1]:>6.3f} {p[2]:>6.3f}')

추론:   0%|          | 0/1 [00:00<?, ?it/s]


[샘플 추론]
text                                      p_pos  p_neg  p_neu
-----------------------------------------------------------------
사이즈가 딱 맞아요                                0.527  0.039  0.434
너무 작아서 입을 수 없어요                           0.006  0.983  0.011
색감 예뻐요                                    0.966  0.008  0.027
배송이 늦어서 짜증나요                              0.003  0.994  0.003
재질은 괜찮은 편이에요                              0.071  0.137  0.792
실밥이 풀려있고 봉제가 엉망이에요                        0.004  0.989  0.007
기모가 따뜻해요                                  0.919  0.014  0.067
비싸고 별로예요                                  0.006  0.984  0.010
잘 받았어요                                    0.968  0.014  0.018
이쁘고 가성비도 좋아요                              0.966  0.009  0.025


## 7-3b-3. Threshold 자동 결정 (분할 문장 1000건 샘플)

### 절차
1. 분할 문장에서 1000건 무작위 샘플 (측면 매칭된 것 중)
2. KcELECTRA 추론 → `max_prob = max(p_pos, p_neg)`
3. 분위수 기반 threshold 결정:
   - **THRESHOLD_HIGH** = max_prob 분포의 75% 분위수
   - **THRESHOLD_LOW**  = max_prob 분포의 25% 분위수
4. 적용:
   - max_prob > HIGH → 모델 예측 (강한 신호)
   - max_prob < LOW  → neutral (약한 신호)
   - 그 외 → 모델 예측 (중간)

### 분위수 기반의 의미
- 약 25%가 neutral로 분류됨 (자연스러운 mixed/약한 표현)
- 강한 표현 25%는 확실히 긍/부
- 중간 50%는 모델 예측 신뢰


In [26]:
# 분할 문장 + 측면 매칭된 데이터 로드
df_pairs = pd.read_parquet(PAIRS_PATH)
print(f'(문장, 측면) 페어 수: {len(df_pairs):,}')
print(f'고유 문장 수: {df_pairs[["리뷰번호", "sent_id"]].drop_duplicates().shape[0]:,}')

# 1000건 샘플 (고유 문장 단위)
N_SAMPLE = 1000
df_unique_sents = df_pairs[['리뷰번호', 'sent_id', 'sentence']].drop_duplicates().reset_index(drop=True)
sample_sents = df_unique_sents.sample(N_SAMPLE, random_state=42).reset_index(drop=True)

print(f'\nthreshold 결정용 샘플: {len(sample_sents)}건 (고유 문장)')

# 추론
print('\n샘플 추론 중...')
sample_probs = predict_sentiment_probs(sample_sents['sentence'].tolist(), batch_size=64)
sample_sents['p_pos'] = sample_probs[:, 0]
sample_sents['p_neg'] = sample_probs[:, 1]
sample_sents['p_neu'] = sample_probs[:, 2]
sample_sents['max_pos_neg'] = np.maximum(sample_probs[:, 0], sample_probs[:, 1])

(문장, 측면) 페어 수: 1,576,839
고유 문장 수: 960,061

threshold 결정용 샘플: 1000건 (고유 문장)

샘플 추론 중...


추론:   0%|          | 0/16 [00:00<?, ?it/s]

In [27]:
# max_prob 분포 분석
max_probs = sample_sents['max_pos_neg'].values

print('[max(p_pos, p_neg) 분포]')
print(f'  평균  : {max_probs.mean():.3f}')
print(f'  표준편차: {max_probs.std():.3f}')
print(f'  최소  : {max_probs.min():.3f}')
print(f'  10%분위: {np.percentile(max_probs, 10):.3f}')
print(f'  25%분위: {np.percentile(max_probs, 25):.3f}')
print(f'  50%분위: {np.percentile(max_probs, 50):.3f}')
print(f'  75%분위: {np.percentile(max_probs, 75):.3f}')
print(f'  90%분위: {np.percentile(max_probs, 90):.3f}')
print(f'  최대  : {max_probs.max():.3f}')

# Threshold 결정 — 분위수 기반
THRESHOLD_LOW  = np.percentile(max_probs, 25)
THRESHOLD_HIGH = np.percentile(max_probs, 75)

# 안전 가드 — 너무 극단적이지 않게 클립
THRESHOLD_LOW  = float(np.clip(THRESHOLD_LOW,  0.40, 0.55))
THRESHOLD_HIGH = float(np.clip(THRESHOLD_HIGH, 0.65, 0.80))

print(f'\n[최종 threshold]')
print(f'  THRESHOLD_LOW  = {THRESHOLD_LOW:.3f} → 이 미만이면 neutral 처리')
print(f'  THRESHOLD_HIGH = {THRESHOLD_HIGH:.3f} → 이 초과면 강한 신호')

[max(p_pos, p_neg) 분포]
  평균  : 0.755
  표준편차: 0.286
  최소  : 0.059
  10%분위: 0.236
  25%분위: 0.583
  50%분위: 0.917
  75%분위: 0.967
  90%분위: 0.978
  최대  : 0.993

[최종 threshold]
  THRESHOLD_LOW  = 0.550 → 이 미만이면 neutral 처리
  THRESHOLD_HIGH = 0.800 → 이 초과면 강한 신호


In [28]:
# 라벨 결정 함수
def assign_labels(probs_array, threshold_low):
    """신뢰도 기반 중립 규칙 적용.

    probs_array: shape (N, 3) — [p_pos, p_neg, p_neu]

    Returns:
        list[str] of labels in {'positive', 'negative', 'neutral'}
    """
    p_pos = probs_array[:, 0]
    p_neg = probs_array[:, 1]
    max_pn = np.maximum(p_pos, p_neg)

    labels = np.where(max_pn < threshold_low, 'neutral',
              np.where(p_pos > p_neg, 'positive', 'negative'))
    return labels


# 샘플에 적용
sample_sents['sentiment'] = assign_labels(sample_probs, THRESHOLD_LOW)

print('[샘플 1000건 분포]')
print(sample_sents['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

print('\n[샘플 결정 예시 — confidence별]')
display_cols = ['sentence', 'p_pos', 'p_neg', 'p_neu', 'sentiment']
print('\n[강한 긍정 예시]')
print(sample_sents.nlargest(5, 'p_pos')[display_cols].to_string(index=False))
print('\n[강한 부정 예시]')
print(sample_sents.nlargest(5, 'p_neg')[display_cols].to_string(index=False))
print('\n[중립 예시 (max_pos_neg 낮음)]')
print(sample_sents.nsmallest(5, 'max_pos_neg')[display_cols].to_string(index=False))

[샘플 1000건 분포]
sentiment
positive    62.3%
neutral     23.3%
negative    14.4%
Name: proportion, dtype: object

[샘플 결정 예시 — confidence별]

[강한 긍정 예시]
                    sentence    p_pos    p_neg    p_neu sentiment
              사이즈 잘 맞고, 이뻐요. 0.989423 0.003515 0.007063  positive
        옷이 도톰하고 기모가 있어 따뜻해여. 0.988814 0.002940 0.008246  positive
        허리가 잘 맞아서 너무 마음에 들어요 0.987891 0.003630 0.008479  positive
           이뻐서 요즘 이거만 입고다녀요. 0.987506 0.004246 0.008247  positive
겨울에 폭닥하게 잘 입고 다닐 수 있을 것 같아요. 0.987361 0.002531 0.010108  positive

[강한 부정 예시]
                sentence    p_pos    p_neg    p_neu sentiment
                많이클줄알았는데 0.002276 0.993115 0.004609  negative
                  너무 커요. 0.002934 0.992631 0.004435  negative
예약배송이 자꾸자꾸 밀려서 한 달 기다렸어요 0.003547 0.992113 0.004340  negative
            배송은 3일 걸렸어요. 0.003009 0.991821 0.005170  negative
   제 키에는 조금 작은 느낌이라 아쉽네요 0.002719 0.990064 0.007217  negative

[중립 예시 (max_pos_neg 낮음)]
                     sentence    p_pos    p_ne

In [29]:
# Threshold 저장
threshold_info = {
    'model_id': MODEL_ID,
    'threshold_low': THRESHOLD_LOW,
    'threshold_high': THRESHOLD_HIGH,
    'sample_size': N_SAMPLE,
    'distribution_stats': {
        'mean': float(max_probs.mean()),
        'std': float(max_probs.std()),
        'p25': float(np.percentile(max_probs, 25)),
        'p50': float(np.percentile(max_probs, 50)),
        'p75': float(np.percentile(max_probs, 75)),
    },
    'sample_label_distribution': sample_sents['sentiment'].value_counts(normalize=True).to_dict(),
}

th_path = OUTPUT_DIR + 'step7_3b_thresholds.json'
with open(th_path, 'w', encoding='utf-8') as f:
    json.dump(threshold_info, f, ensure_ascii=False, indent=2)
print(f'저장: {th_path}')

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_3b_thresholds.json


## 7-3b-4. 트랙 A: ABSA 메인 추론

### 효율 최적화
156만 페어 중 **고유 문장은 ~99.6만**(평균 1.6 측면). 같은 문장을 여러 측면에 대해 반복 추론하면 60% 낭비.

→ **고유 문장 단위로 추론 후 페어에 join**.

### 시간 추정
- 고유 문장 ~99.6만 × T4 GPU batch=64 → **~2-3시간**
- 30분 단위로 중간 저장 권장 (Colab 끊김 대비)

In [30]:
# 고유 문장 추출
df_unique = (
    df_pairs[['리뷰번호', 'sent_id', 'sentence']]
    .drop_duplicates(subset=['리뷰번호', 'sent_id'])
    .reset_index(drop=True)
)
print(f'전체 페어: {len(df_pairs):,}')
print(f'고유 문장: {len(df_unique):,} (절감률: {(1 - len(df_unique)/len(df_pairs))*100:.1f}%)')

전체 페어: 1,576,839
고유 문장: 960,061 (절감률: 39.1%)


In [31]:
# 중간 저장 + 재개 가능 추론
# 30분마다 체크포인트 저장 → Colab 끊김 시 재개

CHECKPOINT_PATH = OUTPUT_DIR + 'step7_3b_track_a_checkpoint.parquet'
SAVE_EVERY = 30000  # 3만 문장마다 중간 저장

# 재개 로직
start_idx = 0
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_parquet(CHECKPOINT_PATH)
    start_idx = len(df_done)
    print(f'체크포인트 발견: {start_idx:,}건 완료. 재개합니다.')
else:
    df_done = pd.DataFrame()
    print('체크포인트 없음. 처음부터 시작합니다.')

print(f'시작 인덱스: {start_idx:,} / {len(df_unique):,}')

체크포인트 발견: 995,743건 완료. 재개합니다.
시작 인덱스: 995,743 / 960,061


In [32]:
# 메인 추론 루프
BATCH_SIZE = 256
texts_all = df_unique['sentence'].tolist()
N = len(texts_all)

# 결과 저장용 (이미 완료된 부분은 df_done에 있음)
result_chunks = [df_done] if len(df_done) > 0 else []

t_start = time.time()

for chunk_start in range(start_idx, N, SAVE_EVERY):
    chunk_end = min(chunk_start + SAVE_EVERY, N)
    chunk_texts = texts_all[chunk_start:chunk_end]

    print(f'\n청크 {chunk_start:,} ~ {chunk_end:,} ({chunk_end-chunk_start:,}건) 추론...')
    chunk_probs = predict_sentiment_probs(chunk_texts, batch_size=BATCH_SIZE)

    # 청크 결과 DataFrame
    chunk_df = df_unique.iloc[chunk_start:chunk_end].copy()
    chunk_df['p_pos'] = chunk_probs[:, 0]
    chunk_df['p_neg'] = chunk_probs[:, 1]
    chunk_df['p_neu'] = chunk_probs[:, 2]
    chunk_df['sentiment'] = assign_labels(chunk_probs, THRESHOLD_LOW)

    result_chunks.append(chunk_df)

    # 체크포인트 저장
    df_done = pd.concat(result_chunks, ignore_index=True)
    df_done.to_parquet(CHECKPOINT_PATH, index=False)

    elapsed_min = (time.time() - t_start) / 60
    progress = chunk_end / N * 100
    eta_min = elapsed_min / max(chunk_end - start_idx, 1) * (N - chunk_end)
    print(f'  진행: {progress:.1f}% | 경과: {elapsed_min:.1f}분 | ETA: {eta_min:.1f}분 | 체크포인트 저장')

print(f'\n전체 추론 완료: {(time.time()-t_start)/60:.1f}분')
print(f'   결과: {len(df_done):,}건')


전체 추론 완료: 0.0분
   결과: 995,743건


In [33]:
# 페어와 join — 한 문장 결과를 그 문장의 모든 측면 페어에 적용
df_unique_pred = df_done[['리뷰번호', 'sent_id', 'p_pos', 'p_neg', 'p_neu', 'sentiment']]

df_aspect_sentiment = df_pairs.merge(
    df_unique_pred,
    on=['리뷰번호', 'sent_id'],
    how='left',
)

# 누락 체크
n_missing = df_aspect_sentiment['sentiment'].isna().sum()
assert n_missing == 0, f'⚠️ 매칭 안 된 페어 {n_missing}건 — 디버깅 필요'
print(f'페어 join 완료: {len(df_aspect_sentiment):,}건')

# 분포
print(f'\n[ABSA 메인 — 측면별 감성 분포]')
print(df_aspect_sentiment['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

# 저장
absa_path = OUTPUT_DIR + 'step7_3b_aspect_sentiment.parquet'
df_aspect_sentiment.to_parquet(absa_path, index=False)
print(f'\n저장: {absa_path}')
print(f'  컬럼: {df_aspect_sentiment.columns.tolist()}')

페어 join 완료: 1,576,839건

[ABSA 메인 — 측면별 감성 분포]
sentiment
positive    65.4%
neutral     22.2%
negative    12.4%
Name: proportion, dtype: object

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_3b_aspect_sentiment.parquet
  컬럼: ['리뷰번호', 'sent_id', 'pair_id', 'sentence', 'aspect_category', 'aspect_keyword', 'topic_id', 'n_sents', 'p_pos', 'p_neg', 'p_neu', 'sentiment']


## 7-3b-5. 트랙 B: 리뷰 단위 감성 (보조)

### 목적
1. **매칭 실패 62만 문장 보완** — outlier·일반 만족 표현이 ABSA 메인에서 빠진 경우 리뷰 단위로 신호 확보
2. **별점 교차검증** — 7-3a에서 검증한 모델 정확도(F1 0.59)를 전체 리뷰에 일반화
3. **인사이트**: 토픽 단위 감성 (ABSA 결과의 sanity check)

### 입력
`embedding_input.parquet` 전체 (62만 리뷰)

### 시간 추정
T4 GPU batch=64 × 62만 → **~1-1.5시간**

In [34]:
# 리뷰 데이터 로드
df_reviews = pd.read_parquet(REVIEW_PATH)
df_reviews['리뷰내용_clean'] = df_reviews['리뷰내용_clean'].fillna('').astype(str)
df_reviews = df_reviews[df_reviews['리뷰내용_clean'].str.len() >= 5].reset_index(drop=True)

# 클러스터 라벨 join (토픽별 분석을 위해)
cluster_labels = np.load('/content/drive/MyDrive/sparta/tp4/review_analysis/data/step5_1_cluster_labels.npy')
# embedding_input과 cluster_labels 길이 정합 — 7-1과 동일 가정
# (※ embedding_input은 길이 5+ 필터링했으므로 미세 차이 가능. 안전을 위해 별점만 활용)

print(f'리뷰 수: {len(df_reviews):,}')
print(f'\n[평점 분포]')
if '평점' in df_reviews.columns:
    print(df_reviews['평점'].value_counts().sort_index())

리뷰 수: 598,301

[평점 분포]
평점
1.0      1587
2.0      2013
3.0     15953
4.0     85350
5.0    491770
Name: count, dtype: int64


In [35]:
# 리뷰 단위 추론 (체크포인트 저장)
TRACK_B_CHECKPOINT = OUTPUT_DIR + 'step7_3b_track_b_checkpoint.parquet'
SAVE_EVERY_B = 30000

start_idx_b = 0
if os.path.exists(TRACK_B_CHECKPOINT):
    df_done_b = pd.read_parquet(TRACK_B_CHECKPOINT)
    start_idx_b = len(df_done_b)
    print(f'체크포인트 발견: {start_idx_b:,}건 완료. 재개합니다.')
else:
    df_done_b = pd.DataFrame()
    print('체크포인트 없음. 처음부터 시작합니다.')

texts_b = df_reviews['리뷰내용_clean'].tolist()
N_B = len(texts_b)
print(f'전체: {N_B:,} | 시작: {start_idx_b:,}')

체크포인트 발견: 623,380건 완료. 재개합니다.
전체: 598,301 | 시작: 623,380


In [36]:
result_chunks_b = [df_done_b] if len(df_done_b) > 0 else []
t_start_b = time.time()

for chunk_start in range(start_idx_b, N_B, SAVE_EVERY_B):
    chunk_end = min(chunk_start + SAVE_EVERY_B, N_B)
    chunk_texts = texts_b[chunk_start:chunk_end]

    print(f'\n청크 {chunk_start:,} ~ {chunk_end:,}')
    chunk_probs = predict_sentiment_probs(chunk_texts, batch_size=BATCH_SIZE)

    chunk_df = df_reviews.iloc[chunk_start:chunk_end].copy()
    chunk_df['p_pos'] = chunk_probs[:, 0]
    chunk_df['p_neg'] = chunk_probs[:, 1]
    chunk_df['p_neu'] = chunk_probs[:, 2]
    chunk_df['sentiment'] = assign_labels(chunk_probs, THRESHOLD_LOW)

    result_chunks_b.append(chunk_df)
    df_done_b = pd.concat(result_chunks_b, ignore_index=True)
    df_done_b.to_parquet(TRACK_B_CHECKPOINT, index=False)

    elapsed = (time.time() - t_start_b) / 60
    progress = chunk_end / N_B * 100
    eta = elapsed / max(chunk_end - start_idx_b, 1) * (N_B - chunk_end)
    print(f'  진행: {progress:.1f}% | 경과: {elapsed:.1f}분 | ETA: {eta:.1f}분')

print(f'\n리뷰 단위 추론 완료: {(time.time()-t_start_b)/60:.1f}분')

# 최종 저장 (체크포인트와 동일하지만 명확한 파일명으로)
review_sent_path = OUTPUT_DIR + 'step7_3b_review_sentiment.parquet'
df_done_b.to_parquet(review_sent_path, index=False)
print(f'\n저장: {review_sent_path}')


리뷰 단위 추론 완료: 0.0분

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_3b_review_sentiment.parquet


In [37]:
# 별점 교차검증 — 트랙 B 결과
if '평점' in df_done_b.columns:
    print('[리뷰 단위 감성 vs 별점]')
    cross = pd.crosstab(
        df_done_b['평점'].fillna(-1).astype(int),
        df_done_b['sentiment'],
        normalize='index'
    ) * 100
    print(cross.round(1))

    print(f'\n[전체 분포]')
    print(df_done_b['sentiment'].value_counts(normalize=True).map(lambda x: f'{x*100:.1f}%'))

[리뷰 단위 감성 vs 별점]
sentiment  negative  neutral  positive
평점                                    
-1              5.5     17.6      76.9
 1             90.7      3.9       5.4
 2             86.0      7.9       6.1
 3             55.4     14.7      29.9
 4             23.5     15.7      60.9
 5              5.9      7.9      86.2

[전체 분포]
sentiment
positive    80.6%
negative    10.2%
neutral      9.2%
Name: proportion, dtype: object


## 7-3b-6. 두 트랙 일관성 검증

같은 리뷰에 대해 두 트랙 결과가 일관적인지 확인.

### 검증 로직
- 리뷰 X의 ABSA 측면 감성들 (예: size=neg, color=pos)
- 리뷰 X의 리뷰 단위 감성 (예: pos)
- 둘이 어떻게 일치/불일치하는지

In [38]:
# 리뷰별 ABSA 결과 집계 (가장 빈번한 측면 감성)
absa_per_review = (
    df_aspect_sentiment.groupby('리뷰번호')['sentiment']
    .agg(lambda x: x.mode()[0] if len(x) > 0 else 'neutral')
    .reset_index()
    .rename(columns={'sentiment': 'absa_sentiment'})
)

# 리뷰 단위 감성과 join
merged = df_done_b[['리뷰번호', 'sentiment']].rename(
    columns={'sentiment': 'review_sentiment'}
).merge(absa_per_review, on='리뷰번호', how='inner')

print(f'두 트랙 매칭 리뷰: {len(merged):,}건')

# 일치도
agreement = (merged['review_sentiment'] == merged['absa_sentiment']).mean()
print(f'\n[두 트랙 감성 일치율]: {agreement*100:.1f}%')

# 교차표
print(f'\n[리뷰 단위 vs ABSA 다수결 — 행=리뷰단위, 열=ABSA]')
ct = pd.crosstab(merged['review_sentiment'], merged['absa_sentiment'], normalize='index') * 100
print(ct.round(1))

두 트랙 매칭 리뷰: 551,613건

[두 트랙 감성 일치율]: 73.6%

[리뷰 단위 vs ABSA 다수결 — 행=리뷰단위, 열=ABSA]
absa_sentiment    negative  neutral  positive
review_sentiment                             
negative              62.7     21.1      16.2
neutral               21.6     64.9      13.5
positive               6.7     17.3      75.9
